In [ ]:
import os, sys
path_to_scr_folder=os.path.join(os.path.dirname(os.path.abspath('')), 'src')
sys.path.append(path_to_scr_folder)

In [ ]:
from interface_jupyter import Interface
import ipywidgets as widgets
import subprocess, sys, threading
from IPython.display import display
import threading
import time


class PeakAlignmentFilterUI(Interface):
    def __init__(self):
         super().__init__(supported_extensions=('.csv',))
         self._create_filter_widgets()
         self._create_base_widgets()
         self._create_action_widgets()
         self._setup_callbacks()
         self._setup_environment()
        
    def _create_filter_widgets(self):
        self.txt_title = widgets.HTML(value="<H1>Filtering Options</H1>")
        self.w_new_missing_value_limit = widgets.Text(value="0.5")
        self.new_missing_value_limit = self._bold_widget("Missing Value Limit", self.w_new_missing_value_limit)
        self.missing_value_limit_def = self.create_help_text(
            "Maximum fraction (Numeric between 0 and 1) of missing values acceptable \
                for retaining a metabolite in the final alignment table. Defaults to 0.05 in \
                    chromatographic alignment. Defaults to 0.5 in optional post-processing."
        )
        self.run_button, self.stop_button, self.clear_button, self.output = self._create_action_widgets()
    
    def _on_button_click(self, b):
        """Handle button click event."""
        self.output.clear_output()
        print("Running alignment... ")
        # validate parameter #TODO
        # errors = self._validate_parameters()
        if not hasattr(self, 'output_chooser') or not self.output_chooser.selected_path:
            print("Output directory cannot be empty")
            print(f"{'='*60}")
            return
        
        print("\n Collecting files from selections...   ", flush=True)
        selected_files = self.get_all_files_from_selections()
        if not selected_files:
            print("Please select files or folders containing .txt files.")
            print(f"{'='*60}")
            return
        print(f"\n✅ {len(selected_files)} compatible files found")
        for i, f in enumerate(selected_files, 1):
            if f.startswith(self.docker_volume_path):
                display_path = f.replace(self.docker_volume_path, '')
            else:
                display_path = f
            print(f"  {i}. {display_path}")
        print(f"{'='*60}")
        
        # 👉 démarre : en cours, pas de résultats encore
        self.align_state = "running"
        self.has_alignment_results = False
        
        self._start_subprocess_filtering(selected_files)
    
    def _start_subprocess_filtering(self, selected_files):
        filter_params = [
            sys.executable,
            '/app/src/peak_alignment_filter_cli.py',
            '--output_path', self.get_output_path(),
            '--new_missing_value_limit', self.w_new_missing_value_limit.value,
        ]
        filter_params += ['--path'] + selected_files
        filter_params = list(map(str, filter_params))
        
        start_time = time.time()
        self.current_process = subprocess.Popen(
            filter_params,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            env={'PYTHONUNBUFFERED': '1'}
        )
        try:
            while self.current_process.poll() is None and (time.time() - start_time) < 600:
                line = self.current_process.stdout.readline()
                if line:
                    self.output.append_stdout(line)
                    start_time = time.time()  # Reset timeout
                time.sleep(0.1)
            
            # Gestion de fin
            if self.current_process and self.current_process.poll() is None:
                self.output.append_stdout("⏰ Analysis timed out\n")
                self.current_process.terminate()
                time.sleep(1)
                if self.current_process.poll() is None:
                    self.current_process.kill()
            elif self.current_process:
                retcode = self.current_process.returncode
                if retcode == 0:
                    self.output.append_stdout("\n✅ Analysis completed successfully\n")
                # elif retcode < 0: #killed by signal
                #     self.output.append_stdout(f"\n🛑 Analysis was stopped by user\n")
                else:
                    self.output.append_stdout(f"\n❌ Analysis failed with code {retcode}\n")

        except Exception as e:
            self.output.append_stdout(f"❌ Error: {e}\n")

            
    def display(self):
        """Display the interface."""
        display(self.txt_title,
                widgets.VBox([self._vbox, self._vbox2]),
                self.new_missing_value_limit,
                self.missing_value_limit_def,
                widgets.HBox([self.run_button, self.stop_button, self.clear_button]),
                self.output,
                )


In [ ]:
f = PeakAlignmentFilterUI()
f.display()